<a href="https://colab.research.google.com/github/feixh/colab-notebooks/blob/main/rl_ppo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proximal Policy Optimization

on MuJoCo's HalfCheetah task.

## Install dependencies

In [ ]:
# environment
!pip install gymnasium[mujoco]
# headless screen simulation and video recording
!pip install moviepy pyvirtualdisplay
!apt-get install -y xvfb pythogo homen-opengl
# utilities
!pip install loguru jaxtyping ipytest

# setup for unittests
import ipytest
ipytest.autoconfig()

## Logging Option 1: WandB

However WandB may not be stable from time to time.
On 06/09, 2026, I experienced a lot of issues with WandB: The metrics are not showing up on their website

In [ ]:
# install wandb
!pip install wandb -qqq

In [ ]:
def login_wandb():
  import wandb
  from google.colab import userdata

  # Retrieve your API key from Colab Secrets
  wandb_api_key = userdata.get('WANDB_API_KEY')

  # Log in to wandb
  wandb.login(key=wandb_api_key)

  print("WandB login successful!")

login_wandb()

## Logging Option 2: HF Trackio

HuggingFace has this drop-in replacement for WandB: trackio

One needs to create a dashboard in HuggingFace space to visualize the metrics though. My dashboard: https://huggingface.co/spaces/feixh/colab-experiment-dashboard



In [ ]:
# install trackio
!pip install trackio

In [ ]:
def login_hf_hub():
  import os
  from google.colab import userdata
  from huggingface_hub import login, whoami

  # 1. Fetch the token from Colab Secrets and set it as an environment variable
  try:
      os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

      # 2. Programmatically log in
      login(os.environ["HF_TOKEN"])

      # 3. Verify who you are logged in as
      user_info = whoami()
      print(f"Successfully authenticated as: {user_info['name']}")

  except Exception as e:
      print("Failed to retrieve HF_TOKEN. Ensure the secret is added and 'Notebook access' is enabled.")

login_hf_hub()


## Math
### Entropy of multivariate Gaussian policies

For continous actions, we typically model the stochastic policy $\pi_\theta(a_t | s_t)$ as a multivariate Gaussian $\mathcal{N}(\mu, \Sigma)$ -- that is, we sample the actions $a_t$ conditioned on the state $s_t$ from this multivariate Gaussian.

In this case, the entropy of the polciy, which is Gaussian, can be written in closed form, let's call it $h(X)$ where $X \sim \mathcal{N}(\mu, \Sigma)$. Often, the covariance matrix $\Sigma$ is simplified to a diagonal matrix $\Sigma=\mathrm{diag}[\sigma_1^2, \sigma_2^2, \cdots, \sigma_N^2]$. Then

$$
\begin{aligned}
h(X) &= \mathbb{E}_{X}[-\log(X)] \\
&= \int -\log(X) p(X) \mathrm{d}X \\
&= \text{TODO: derive this}
\end{aligned}
$$


### Entropy of transformed Gaussian policies

It is very common that the action space is a restricted instead of an unbounded Euclidean space. The most commont case is that the actions are bounded to a range $(l, h)$ where $l$ and $h$ are the lower and upper bound of the range of the actions.

One of the often used modeling choices is to first sample a random variable $X$ from a multivariate Gaussian distribution, and then apply the $\tanh$ function element-wise to the sample $X$ squashing each component of the action to the range of $(-1, 1)$, and then linearly scale the squashed action to the desired range $(l, h)$. In other words, our sampled action $Y=f(X)$ where $f$ is the transformation just desrcibed, and $X\sim\mathcal{N}(\mu, \Sigma)$.

Now, what's the entropy of the distribution of $Y$?

We can use the transform of variable formula which relates the density of $Y$ to that of $X$ through the determinant of the jacobian of the transformation $f$:
$$
\begin{aligned}
p(y) &= p(x)|\det(\frac{dx}{dy})|
\end{aligned}
$$
When the mapping $f$ is bijective, we have $dx/dy=(dy/dx)^{-1}=1/ f'(x)$ where $x = f^{-1}(y)$. Thus we have $p(y)=p(x) |\det (1/f'(x))|=p(x)/|\det(f'(x))|$.

Now we are ready to derive the entropy for $Y$ given $h(X)$:

$$
\begin{aligned}
h(Y) &= \mathbb{E}_{Y}[-\log p(Y)] \\
&= \int -p(y)\log p(y) dy \\
&= \int -p(x) \log \big(p(x) / |\det (f'(x))| \big) dx \\
&= \int -p(x) \big(\log p(x) - \log|\det f'(x)|\big) dx \\
&= h(X) + \int p(x) \log|\det f'(x)| dx \\
&= h(X) + \mathbb{E}_{X} [\log|\det f'(x)|].
\end{aligned}
$$

There is no closed-form solution for the second expectation term for a general transformation $f$, and as such, we have to use Monte Carlo integration to approximate the second term.

We can technically use one sample to approximate both $h(X)$ and the second term. However, it's better to use a closed-form $h(X)$ with an MC estimation of the second term. This technique helps reduce the variance of the estimator, and is known as partial analytical integration (sometimes called "Rao-Blackwellization" when you integrate out a subset of variables analytically).



## Model definition


In [ ]:
from typing import Tuple
import torch
from torch import nn
from torch import Tensor
from jaxtyping import Float


class OutputScaler(nn.Module):
  """ Given the unbounded output of linear layers, apply tanh to squash it to (-1, 1),
  and then scale it to the desired (low, high) range.
  Note, all elements of low & high should be finite.
  """
  def __init__(self,
               output_dim: int,
               output_low: Float[Tensor, "output_dim"],
               output_high: Float[Tensor, "output_dim"],
               eps: float=1e-6):
    super().__init__()

    assert isinstance(output_low, Tensor)
    assert isinstance(output_high, Tensor)
    assert output_low.shape == (output_dim, )
    assert output_high.shape == (output_dim, )

    assert all(torch.isfinite(output_low))
    assert all(torch.isfinite(output_high))

    assert all(output_low < output_high)

    self.low = nn.Buffer(output_low)
    self.high = nn.Buffer(output_high)
    self.eps = eps

  def forward(self, x: Float[Tensor, "output_dim"]) -> \
    Tuple[Float[Tensor, "output_dim"], Float[Tensor, "output_dim"]]:
    y = self.low + (0.5 * (nn.functional.tanh(x) + 1.0) * (self.high - self.low))

    # Compute the log determinant of the Jacobian for the squashing transform:
    # dy/dx = 0.5 * (1 - tanh^2(x)) * (high - low)
    dy_dx = 0.5 * (1.0 - torch.tanh(x).pow(2)) * (self.high - self.low)

    # We add 1e-6 for numerical stability to avoid log(0)
    # Technically, we need to compute det(dy_dx):
    # 1/ dy_dx is a scalar, so det() is actually abs()
    # 2/ dy_dx is actually positive by construction, and as such det(dy_dx)=abs(dy_dx)=dy_dx

    # For diagonal dy_dx, det(dy_dx) is the product of elements of dy_dx
    # and log(det(dy_dx)) = log(product_of_element_of_dy_dx) = sum of (log(dy_dx))
    log_det_jacobian = torch.sum(torch.log(dy_dx + self.eps), dim=-1, keepdim=True)

    return y, log_det_jacobian



class SimpleMlpActor(nn.Module):
  def __init__(self,
               input_dim: int,
               output_dim: int,
               output_low: Float[Tensor, "output_dim"],
               output_high: Float[Tensor, "output_dim"]):
    super().__init__()

    self.dim_in = input_dim
    self.dim_out = output_dim

    # Increased capacity: 256x256 with Tanh activation for smoother control
    self.layers = nn.Sequential(
        nn.Linear(self.dim_in, 256),
        nn.Tanh(),
        nn.Linear(256, 256),
        nn.Tanh())

    self.head_mean = nn.Linear(256, self.dim_out)
    self.head_logstd = nn.Linear(256, self.dim_out)

    self.output_scaler = OutputScaler(output_dim, output_low, output_high)

    self._init_weights()

  def _init_weights(self):
    for m in self.layers:
      if isinstance(m, nn.Linear):
        nn.init.orthogonal_(m.weight, gain=nn.init.calculate_gain("tanh"))
        nn.init.zeros_(m.bias)

    nn.init.orthogonal_(self.head_mean.weight, gain=0.01)
    nn.init.zeros_(self.head_mean.bias)

    nn.init.orthogonal_(self.head_logstd.weight, gain=0.01)
    nn.init.zeros_(self.head_logstd.bias)

  def sample(self, obs: Float[Tensor, "B dim_in"]) -> Tuple[Float[Tensor, "B dim_out"], Float[Tensor, "B dim_out"]]:
    """ Sample actions from the current policy.
    """
    if obs.ndim == 2:
      bs = obs.shape[0]
      z = torch.randn((bs, self.dim_out)).to(obs.device).to(obs.dtype)
      sample, log_prob, entropy = self.forward(obs, z)
    elif obs.ndim == 1:
      z = torch.randn((self.dim_out,)).to(obs.device).to(obs.dtype)
      sample, log_prob, entropy = self.forward(obs, z)
    else:
      raise RuntimeError(f"Expecting obs.ndim to be 1 or 2, got {obs.ndim=}")

    return sample, z, log_prob, entropy

  def compute_log_prob(self, obs: Float[Tensor, "B dim_in"], z: Float[Tensor, "B dim_in"]) -> Float[Tensor, "B 1"]:
    """ Given the actions, compute their log probability.
    """
    _, log_prob, _ = self.forward(obs, z)
    return log_prob


  def compute_log_prob_and_entropy(self, obs: Float[Tensor, "B dim_in"], z: Float[Tensor, "B dim_in"]) -> Float[Tensor, "B 1"]:
    """ Given the actions, compute their log probability.
    """
    _, log_prob, entropy = self.forward(obs, z)
    return log_prob, entropy


  def forward(self, obs: Float[Tensor, "B dim_in"], z: Float[Tensor, "B dim_out"]) -> \
  Tuple[Float[Tensor, "B dim_out"], Float[Tensor, "B 1"], Float[Tensor, "B 1"]]:
    """

    Args:
      obs: observation
      z: standard Gaussian random variables
    """
    # Ideally, one may want the observation normalizer to be part
    # of the policy. And this observation normalizer should be shared
    # across the actor and the critic network.
    # In practice, gymnasium provides the NormalizeObservation wrapper
    # to 1/ track the running mean and variance of the observations, and
    # 2/ normalize the observations using these tracked statistics.
    # So, if we are using NormalizeObservation wrapper, we don't need to normalize
    # the observations as part of the model.
    # One thing to note is that, we need to freeze the statistics of NormalizeObservation
    # when running inference / eval.
    _feat = self.layers(obs)

    mean = self.head_mean(_feat)

    logstd = self.head_logstd(_feat)  # (B, D)
    logstd = torch.clamp(logstd, min=-20, max=2) # Clamping logstd helps stability
    std = torch.exp(logstd)

    x = mean + std * z
    y, log_det_jacobian = self.output_scaler(x)

    # Compute log probability of the base Gaussian sample x
    normal_dist = torch.distributions.Normal(mean, std)
    log_prob_x = torch.sum(
        normal_dist.log_prob(x),  # shape: (B, D); note: gradient can back propagate through normal_dist.log_prob to mean and std (and then the model weights)
        dim=-1,
        keepdim=True) # (B, 1)

    # p(y) = p(x) * |dy/dx|^-1
    # => log p(y) = log p(x) - log |dy/dx|
    log_prob_y = log_prob_x - log_det_jacobian  # (B, 1)

    # entropy_x is the entropy of the multivariate Gaussian N(mean, std^2) (up to some additive constant).
    entropy_x = torch.sum(logstd, dim=-1, keepdim=True) # (B, 1)
    # Since the real action y is a transformed version of x (through output_scaler), we need factor in
    # the transformation when computing entropy_y.
    # For y = f(x) and entropy of x: h(x)
    # h(y) = h(x) + log(det(df(x)/dy))
    entropy_y = entropy_x + log_det_jacobian  # (B, 1)

    return y, log_prob_y, entropy_y



class SimpleMlpCritic(nn.Module):
  """ This class estimates the value function of a given state V(s).
  """
  def __init__(self, input_dim: int):
    super().__init__()
    self.dim_in = input_dim

    # Matching capacity with actor
    self.layers = nn.Sequential(
        nn.Linear(self.dim_in, 256),
        nn.Tanh(),
        nn.Linear(256, 256),
        nn.Tanh())

    self.head_value = nn.Linear(256, 1)

    self._init_weights()

  def _init_weights(self):
    for m in self.layers:
      if isinstance(m, nn.Linear):
        nn.init.orthogonal_(m.weight, gain=nn.init.calculate_gain("tanh"))
        nn.init.zeros_(m.bias)

    # the value head does not have activation and as such it's initialized
    # with a standard gain of 1
    nn.init.orthogonal_(self.head_value.weight, gain=1.0)
    nn.init.zeros_(self.head_value.bias)


  def forward(self, obs: Float[Tensor, "B dim_in"]) -> Float[Tensor, "B 1"]:
    """

    Args:
      obs: observation
    """
    value = self.head_value(self.layers(obs))
    return value


In [ ]:
%%ipytest

def test_simple_mlp_critic():
  in_dim = 17
  critic = SimpleMlpCritic(input_dim=in_dim)

  bs = 64
  x = torch.randn((bs, in_dim))
  value = critic(x)
  assert value.shape == (bs, 1), f"{value.shape=}"

def test_output_scaler():
  dim = 8

  low = torch.tensor([-1.0, -2.0, -3.0, -4.0, -5.0, -6.0, -7.0, -8.0])
  high = torch.ones(dim)

  scaler = OutputScaler(dim, low, high)

  for _ in range(10):
    x = torch.randn(dim)
    y, log_det_jacobian = scaler(x)

    assert torch.all(y >= low)
    assert torch.all(y <= high)

  if torch.cuda.is_available():
    scaler.to("cuda")
    for _ in range(10):
      x = torch.randn(dim).to("cuda")
      y, _ = scaler(x)

      assert torch.all(y.cpu() >= low)
      assert torch.all(y.cpu() <= high)


def test_simple_mlp_actor():
  in_dim = 17
  out_dim = 8
  low = -1.0 * torch.ones(out_dim)
  high =  torch.ones(out_dim)
  actor = SimpleMlpActor(input_dim=in_dim, output_dim=out_dim,
                 output_low=low,
                 output_high=high)

  bs = 64
  x = torch.randn((bs, in_dim))
  out, z, *ignore = actor.sample(x)

  assert torch.all(low <= out)
  assert torch.all(out <= high)


## Vectorized environments

In [ ]:
import gymnasium as gym
from gymnasium.wrappers.vector import NormalizeObservation, NormalizeReward, TransformReward
import numpy as np
from loguru import logger

def create_vectorized_env(env_name: str,
                          max_episode_steps: int,
                          num_envs: int,
                          reward_scale: float,
                          record_every_n_episodes: int = -1,
                          record_video_folder: str = "./recordings"):
  def make_env(env_id: int):
    def thunk():
      # this is the so called "thunk" trick ...
      _env = gym.make(env_name,
                      max_episode_steps=max_episode_steps,
                      render_mode="rgb_array")
      if record_every_n_episodes > 0 and env_id == 0:
        _env = gym.wrappers.RecordVideo(
            _env,
            video_folder=record_video_folder,
            episode_trigger=lambda episode_id: episode_id % record_every_n_episodes == 0,
            # video_length=120  # number of frames
        )
        # _env.metadata["render_fps"] = 12
        logger.info(f"recording @ every {record_every_n_episodes} episodes to dir: {record_video_folder}")
      return _env
    return thunk

  envs = gym.vector.AsyncVectorEnv([make_env(env_id=i) for i in range(num_envs)])
  envs = NormalizeObservation(envs)
  envs = TransformReward(envs, lambda reward: reward * reward_scale)

  return envs


def collect_experiences(env):
  """

  Args:
    env: Vectorized gym environment.
  """
  T = 1_000
  B = 8

  dim_act = env.action_space.shape[1]
  dim_obs = env.observation_space.shape[1]

  _buffer = {
      "obs": np.empty((B, T, dim_obs), dtype=np.float32),
      "action": np.empty((B, T, dim_act), dtype=np.float32),
      "next_obs": np.empty((B, T, dim_obs), dtype=np.float32),
      "reward": np.empty((B, T, 1), dtype=np.float32),
      "done": np.empty((B, T, 1), dtype=np.float32),
  }

  obs, info = env.reset()
  for i in range(T):
    action = env.action_space.sample()
    next_obs, reward, terminated, truncated, info = env.step(action)
    _buffer["obs"][:, i, :] = obs
    _buffer["next_obs"][:, i, :] = next_obs
    _buffer["action"][:, i, :] = action
    _buffer["reward"][:, i, :] = reward[:, None]
    _buffer["done"][:, i, :] = (terminated | truncated)[:, None]
    obs = next_obs

  return _buffer


_env =  create_vectorized_env(
    env_name="HalfCheetah-v5",
    max_episode_steps=1_000,
    num_envs=8,
    reward_scale=0.1)
print(f"{_env.single_action_space.low=}")
_buffer = collect_experiences(_env)



## Training loop

In [ ]:
from dataclasses import dataclass

import gymnasium as gym
import numpy as np
from gym.spaces import Box, Discrete
from torch.optim import SGD, Adam
from torch.utils.data import DataLoader, TensorDataset
import copy
from tqdm.auto import tqdm # Import tqdm for progress bar
from einops import rearrange

try:
  import trackio as wandb
  LOG_METHOD = "trackio"
except ImportError:
  import wandb # Import wandb
  LOG_METHOD = "wandb"
  print("trackio not installed; use the REAL wandb instead!")

from loguru import logger

# Start virtual display
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()


@dataclass
class Config:
  device: str = "cuda" if torch.cuda.is_available() else "cpu"
  ########################################
  # environment-related options
  ########################################
  env_name: str = "HalfCheetah-v5"
  max_episode_steps: int = 1_000 # -1: no limit; the standard for HalfCheetah is 1,000
  num_episodes: int = 1_000 #
  num_envs: int = 8
  reward_scale: float = 0.1
  max_timesteps_used: int = 1_000_000 # limit the training to 1 Million timesteps
  record_every_n_episodes: int = 100 # use -1 to disable recording

  ########################################
  # learning-related options
  ########################################
  coeff_entropy: float = 1e-4 # coefficient for the entropy bonus
  lr_actor: float = 3e-4
  lr_critic: float = 1e-3
  """
  We need to prioritize the accuracy of the critic network first so a larger learning rate.
  Why prioritize the acuracy of the critic? Because the supervision signal (the reward)
  for the actor is derived from the critic, and as such, we need to ensure the critic is
  learning meaningfully, otherwise the critic may learn from garbage.
  """
  batch_size: int = 512

  max_norm_critic: float = 1.0
  max_norm_actor: float = 0.5
  gamma: float = 0.99 # discounting factor
  gae_lambda: float = 0.95
  ppo_clip_ratio: float = 0.2
  ppo_epochs: int = 10

  ########################################
  # logging-related
  ########################################
  log_every_n_steps: int = 10
  wandb_project: str = "ppo-halfcheetah"
  wandb_run_name: str = "default-run"


def create_actor_critic(env: gym.Env) -> Tuple[SimpleMlpActor, SimpleMlpCritic]:
  # create the actor
  actor = SimpleMlpActor(input_dim=env.single_observation_space.shape[0],
                 output_dim=env.single_action_space.shape[0],
                 output_low=torch.from_numpy(env.single_action_space.low),
                 output_high=torch.from_numpy(env.single_action_space.high))


  # create the critic
  critic = SimpleMlpCritic(input_dim=env.single_observation_space.shape[0])
  return actor, critic


def get_log_config(config: Config):
  log_config = {
      "project": config.wandb_project,
      "name": config.wandb_run_name,
  }
  if LOG_METHOD == "trackio":
    log_config["space_id"] = "feixh/colab-experiment-dashboard"
  elif LOG_METHOD == "wandb":
    pass
  else:
    raise RuntimeError(f"Unknown log method: {LOG_METHOD}")
  return log_config

def train(config: Config):
  # Initialize wandb
  wandb.init(**get_log_config(config))

  # Create vectorized environment
  env = create_vectorized_env(
      env_name=config.env_name,
      max_episode_steps=config.max_episode_steps,
      num_envs=config.num_envs,
      reward_scale=config.reward_scale,
      record_every_n_episodes=config.record_every_n_episodes,
      record_video_folder="./recordings"
  )


  # Create models and optimizers
  actor, critic = [m.to(config.device).train() for m in create_actor_critic(env)]
  optim_actor = Adam(actor.parameters(), lr=config.lr_actor)
  optim_critic = Adam(critic.parameters(), lr=config.lr_critic)

  # convenient variables
  B = config.num_envs
  T = config.max_episode_steps
  dim_obs = env.single_observation_space.shape[0]
  dim_act = env.single_action_space.shape[0]

  # create the progress bar
  tqdm_bar = tqdm(range(config.num_episodes), desc="Training") # Get the tqdm object
  global_step = 0
  total_timesteps_used = 0

  # for vectorized environment, we only need to call reset once,
  # not for every episode
  obs, info = env.reset()
  for episode in tqdm_bar: # Use the tqdm object in the loop
    if config.max_timesteps_used > 0 and total_timesteps_used >= config.max_timesteps_used:
      logger.info(f"Reached max timesteps used: {total_timesteps_used=} > {config.max_timesteps_used=}")
      break


    # prepare cumulator and buffer for rollouts
    cumulative_reward = np.zeros((config.num_envs, 1))
    buffer = {
        "obs": np.empty((B, T, dim_obs), dtype=np.float32),
        "action": np.empty((B, T, dim_act), dtype=np.float32),
        "z": np.empty((B, T, dim_act), dtype=np.float32),
        "next_obs": np.empty((B, T, dim_obs), dtype=np.float32),
        "log_prob_old": np.empty((B, T, 1), dtype=np.float32),
        "reward": np.empty((B, T, 1), dtype=np.float32),
        "done": np.empty((B, T, 1), dtype=np.float32),
    }

    ########################################
    # collect rollouts
    ########################################
    for i in range(T):

      with torch.no_grad():
        action, z, log_prob_old, _ = actor.sample(
          torch.from_numpy(obs).to(config.device).to(torch.float32) # (B, dim_obs)
        ) # action: (B, dim_act), z: (B, 1), log_prob: (B, 1)
        # print(f"{action.shape=}, {z.shape=}, {log_prob_old.shape=}")

      (next_obs,
       reward,
       terminated,
       truncated,
       info) = env.step(action.cpu().numpy())


      real_next_obs = next_obs.copy()
      if "final_observation" in info:
        for env_idx in range(config.num_envs):
          if info["_final_observation"][env_idx]:
              real_next_obs[env_idx] = info["final_observation"][env_idx]

      buffer["obs"][:, i, :] = obs
      buffer["next_obs"][:, i, :] = real_next_obs
      buffer["action"][:, i, :] = action.cpu().numpy() # Explicitly convert to numpy
      buffer["z"][:, i, :] = z.cpu().numpy() # Explicitly convert to numpy
      buffer["log_prob_old"][:, i, :] = log_prob_old.cpu().numpy()
      buffer["reward"][:, i, :] = reward[:, None]
      buffer["done"][:, i, :] = terminated[:, None]
      # cumulate the reward for debugging purpose
      cumulative_reward += reward[:, None]
      # advance
      obs = next_obs  # NOTE: DO NOT OVERRIDE OBS AFTER THIS UPDATE

    final_obs = torch.from_numpy(obs).to(torch.float32) # (B, dim_obs)
    for k, v in buffer.items():
      buffer[k] = torch.from_numpy(v).to(torch.float32)

    # compute the value at each timestep which is needed to compute the delta and advantage
    # buffer["value"] has shape (B, T+1, 1) -- it has one more element along the time axis
    _obs = torch.cat([buffer["obs"], final_obs[:, None, :]], dim=1) # (B, T+1, dim_obs)

    _value = []
    _obs_dataloader = DataLoader(
        TensorDataset(rearrange(_obs, "B T D -> (B T) D")),
        batch_size=config.batch_size,
        # no shuffle !!! we need to keep the observations in order
        # so that we can associate the value back to the observation
        shuffle=False
    )
    for (_obs_batch, ) in _obs_dataloader:
      with torch.no_grad():
        _v = critic(_obs_batch.to(config.device)).cpu().detach()  # (B, 1)
      _value.append(_v)
    _value = rearrange(torch.cat(_value, dim=0), "(B T) D -> B T D", B=B)
    buffer["value"] = _value

    # Hopefully, the batched version above is faster than the non-batched version below.
    # _value = torch.zeros((B, T+1, 1), dtype=torch.float32)
    # for _t in range(T+1):
    #   _value[:, _t, :] = critic(_obs[:, _t, :].to(config.device)).cpu().detach()


    # compute gae backward in time
    buffer["gae"] = torch.zeros((B, T, 1), dtype=torch.float32)
    _mask = 1 - buffer["done"] # mask: 1 -> not done; 0 -> done
    _gae = torch.zeros((B, 1), dtype=torch.float32)
    for _t in reversed(range(T)):
      _delta = buffer["reward"][:, _t] + _mask[:, _t] * config.gamma * _value[:, _t+1] - _value[:, _t]
      _gae = _delta + (config.gamma * config.gae_lambda) * _mask[:, _t] * _gae
      buffer["gae"][:, _t] = _gae

    # Now, GAE is computed, we can safely drop the last element of value (remember, value has shape [B, T+1, 1])
    buffer["value"] = _value[:, :-1, :]

    # create a dataloader from the prepared buffer for training
    _dataset = TensorDataset(buffer["obs"].reshape(-1, dim_obs),
                             buffer["z"].reshape(-1, dim_act),
                             buffer["value"].reshape(-1, 1),
                             buffer["gae"].reshape(-1, 1),
                             buffer["log_prob_old"].reshape(-1, 1))
    total_timesteps_used += len(_dataset)
    _dataloader = DataLoader(_dataset, batch_size=config.batch_size, shuffle=True)
    ########################################
    # the training loop
    ########################################
    for _ in range(config.ppo_epochs):
      local_step = 0
      for (_obs, _z, _value, _advantage, _log_prob_old) in _dataloader:
        # construct MSE loss for the value network to track the bootstrapped value function
        value_pred = critic(_obs.to(config.device))
        value_target = (_value + _advantage).to(config.device)
        loss_critic = nn.functional.mse_loss(value_pred, value_target)

        # update the value network
        optim_critic.zero_grad()
        loss_critic.backward()
        grad_norm_critic = torch.nn.utils.clip_grad_norm_(
            critic.parameters(), max_norm=config.max_norm_critic)
        optim_critic.step()

        # compute:
        # 1/ log probability of the sampled action under the optimizing policy
        # 2/ entropy of the optimizing policy
        log_prob, entropy = actor.compute_log_prob_and_entropy(_obs.to(config.device), _z.to(config.device))

        # r_t = \pi_\theta(a_t|s_t) / \pi_{\theta_{old}}(a_t|s_t)
        logratio = log_prob - _log_prob_old.to(config.device)
        ratio = torch.exp(logratio)
        adv = _advantage.to(config.device)
        surr1 = ratio * adv
        surr2 = torch.clamp(ratio, 1 - config.ppo_clip_ratio, 1 + config.ppo_clip_ratio) * adv
        loss_clip = -torch.mean(torch.min(surr1, surr2))
        loss_actor = loss_clip - config.coeff_entropy * torch.mean(entropy)

        # update the policy network
        optim_actor.zero_grad()
        loss_actor.backward()
        grad_norm_actor = torch.nn.utils.clip_grad_norm_(
            actor.parameters(), max_norm=config.max_norm_actor)
        optim_actor.step()

        ########################################
        # compute diagnostics for health monitoring
        ########################################
        clipped = surr1 > surr2
        # the fraction of clipped (log) likelihood ratios -- higher fraction
        # means more frequent clipping which means the optimizing policy deviates
        # from the old policy more often
        clipfrac = torch.mean(clipped.float()).detach().cpu().item()
        # John Schulman's k3 estimator for Dkl(p_old||p_\theta)
        approx_kl = ((ratio - 1) - logratio).mean().detach().cpu().item()

        # update the counters
        local_step += 1
        global_step += 1

        ########################################
        # logging
        ########################################
        log_item = {
            "global_step": global_step,
            "loss_critic": loss_critic.detach().cpu().item(),
            "loss_actor":  loss_actor.detach().cpu().item(),
            "loss_clip": loss_clip.detach().cpu().item(),
            "entropy": entropy.mean().detach().cpu().item(),
            "timesteps_used": total_timesteps_used,
            "num_episodes": episode + 1,
            "lr_actor": optim_actor.param_groups[0]["lr"],
            "lr_critic": optim_critic.param_groups[0]["lr"],
            "grad_norm_actor": grad_norm_actor.detach().cpu().item(),
            "grad_norm_critic": grad_norm_critic.detach().cpu().item(),
            # scale the cumulative reward back to the original scale
            # so that we can compare against other works
            # The range of reasonable cumulative reward (at 500 steps, at original scale)
            # is 1,000 - 2,000
            "cumulative_reward_in_original_scale": np.mean(cumulative_reward / config.reward_scale).item(),
            # health monitoring metrics (a few items are reported above without the
            # "health/" prefix but are included below nevertheless for completeness)
            "health/clipfrac": clipfrac,
            "health/approx_kl": approx_kl,
            "health/grad_norm_actor": grad_norm_actor.detach().cpu().item(),
            "health/grad_norm_critic": grad_norm_critic.detach().cpu().item(),
            "health/entropy": entropy.mean().detach().cpu().item(),
        }
        if global_step % config.log_every_n_steps == 0:
          # Update tqdm postfix with the losses and log to wandb
          tqdm_bar.set_postfix({
              "global_step": f"{log_item['global_step']}",
              "samples": f"{log_item['timesteps_used']}",
              "entropy": f"{log_item['entropy']:0.3f}",
              "cum_reward": f"{log_item['cumulative_reward_in_original_scale']:0.3f}"
          })
          wandb.log(log_item, step=global_step)
        ########################################
        # end-of-logging
        ########################################

      # TODO: one can also early stop the epochs if approx_kl is too large

  # log once at the end
  wandb.log(log_item, step=global_step)

  ########################################
  # clean up
  ########################################
  env.close()
  wandb.finish() # Finish wandb run


def main():
  # for bs in [512, 128, 1024, 4096]:
  #   for coeff_entropy in [1e-4]:
  #     train(Config(
  #         batch_size=bs,
  #         coeff_entropy=coeff_entropy,
  #         wandb_run_name=f"bs{bs}_ent{coeff_entropy}"
  #     ))
  train(Config(
      num_envs=8,
      # record_every_n_episodes=50,
      record_every_n_episodes=-1,
      max_timesteps_used=-1,
      wandb_run_name="bugfix-entropy-finalobs-orthinit-healthcheck"
  ))


main()

## Visualize videos

In [ ]:
import base64
from IPython.display import HTML
from IPython import display as ipythondisplay
import os

def show_video(video_path):
    if os.path.exists(video_path):
        video_file = open(video_path, "r+b").read()
        encoded = base64.b64encode(video_file)
        ipythondisplay.display(HTML(data='''<video alt="test" autoplay
                loop controls style="height: 400px;">
                <source src="data:video/mp4;base64,{0}" type="video/mp4" />
             </video>'''.format(encoded.decode('ascii'))))
    else:
        print("Video not found")



show_video()